In [19]:
from pyspark.sql import SparkSession
import pyspark

In [20]:
app = (
    SparkSession.builder
    .appName("Spark Dockerizado")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [21]:
print(app.version)

3.5.1


In [22]:
print(app.sparkContext.defaultParallelism)

12


In [23]:
print(app.conf.get('spark.sql.shuffle.partitions'))

8


In [24]:
import os, requests
from pathlib import Path

In [25]:
directorio = Path('data')
directorio.mkdir(exist_ok=True)

ubicacion = directorio / "yellow_tripdata_202401.parquet"

In [26]:

def load_data(url, out):
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()

    with open(out, 'wb') as archivo:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                archivo.write(chunk)


In [27]:
load_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet', ubicacion)

In [28]:
type(ubicacion)

pathlib.PosixPath

In [29]:
df = app.read.parquet(str(ubicacion))

In [30]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [31]:
df.count()

2964624

In [32]:
df.show(10, truncate=True)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

In [33]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

In [34]:
# quiero obtener dinamicamente las columnas numericas en una lista
#num_cols = [...]

In [35]:
#df.select(*num_cols).describe().show()

In [36]:
# Viajes por dia

In [37]:
from pyspark.sql import functions as F

In [38]:
viajes_diarios = (
    df
    .withColumn('pickup_date', F.to_date(F.col('tpep_pickup_datetime')))
    .groupby('pickup_date')
    .agg(F.count('*').alias('num_trips'))
    .orderBy('pickup_date')
)

In [39]:
viajes_diarios.show()

+-----------+---------+
|pickup_date|num_trips|
+-----------+---------+
| 2002-12-31|        2|
| 2009-01-01|        3|
| 2023-12-31|       10|
| 2024-01-01|    81013|
| 2024-01-02|    75519|
| 2024-01-03|    82427|
| 2024-01-04|   102901|
| 2024-01-05|   103178|
| 2024-01-06|    97117|
| 2024-01-07|    67543|
| 2024-01-08|    80034|
| 2024-01-09|    93962|
| 2024-01-10|    95000|
| 2024-01-11|   105010|
| 2024-01-12|   103655|
| 2024-01-13|   104758|
| 2024-01-14|    94420|
| 2024-01-15|    77033|
| 2024-01-16|    93057|
| 2024-01-17|   110365|
+-----------+---------+
only showing top 20 rows



In [40]:
# Transformacion Silver:
# trip_distance >= 0
# total_amount >= 0
# pickup_ts <= dropoff_ts

# renombrar tpep_pickup_... -> pickup o dropoff



In [41]:
df_silver = (
    df
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_ts')
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_ts')
    .withColumn('pickup_date', F.to_date(F.col('pickup_ts')))
    .withColumn('pickup_month', F.date_format('pickup_ts', 'yyyy-MM'))
    .withColumn('pickup_hour', F.hour('pickup_ts'))
    .withColumn(
        'trip_duration_min',
        (F.unix_timestamp('dropoff_ts') - F.unix_timestamp('pickup_ts')) / 60.0
    )
    .filter(F.col('trip_distance') >= 0)
    .filter(F.col('total_amount') >= 0)
    .filter(F.col('trip_duration_min') >= 0)
    
)

In [42]:
df_silver.show()

+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+------------+-----------+-------------------+
|VendorID|          pickup_ts|         dropoff_ts|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|pickup_date|pickup_month|pickup_hour|  trip_duration_min|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+------------+-----------+-------------------+
|       2|2024-01-01 00:5

In [43]:
df.count() - df_silver.count()

35560

In [45]:
zones_directorio = directorio / 'taxi_zone_lookup.csv'

In [46]:
print(zones_directorio)

data/taxi_zone_lookup.csv


In [47]:
zones_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

In [48]:
load_data(zones_url, zones_directorio)

In [49]:
df_zones = app.read.csv(str(zones_directorio), header=True, inferSchema=True)

In [50]:
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [51]:
df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [52]:
dim_zones_PU = df_zones.select(
    F.col('LocationID').alias('PU_location_id'),
    F.col('Borough').alias('PU_borough'),
    F.col('Zone').alias('PU_zone')
)

In [53]:
dim_zones_DO = df_zones.select(
    F.col('LocationID').alias('DO_location_id'),
    F.col('Borough').alias('DO_borough'),
    F.col('Zone').alias('DO_zone')
)

In [57]:
dim_zones_DO.show(5)

+--------------+-------------+--------------------+
|DO_location_id|   DO_borough|             DO_zone|
+--------------+-------------+--------------------+
|             1|          EWR|      Newark Airport|
|             2|       Queens|         Jamaica Bay|
|             3|        Bronx|Allerton/Pelham G...|
|             4|    Manhattan|       Alphabet City|
|             5|Staten Island|       Arden Heights|
+--------------+-------------+--------------------+
only showing top 5 rows



In [58]:
df_silver_enriched = (
    df_silver
    .join(dim_zones_PU, df_silver['PULocationID'] == dim_zones_PU['PU_location_id'], 'left')
    .join(dim_zones_DO, df_silver['DOLocationID'] == dim_zones_DO['DO_location_id'], 'left')
)

df_silver_enriched = df_silver_enriched.cache()

In [59]:
df_silver_enriched.show(10)

26/02/25 18:48:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 28:==============================================>         (10 + 2) / 12]

+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+------------+-----------+------------------+--------------+----------+--------------------+--------------+----------+--------------------+
|VendorID|          pickup_ts|         dropoff_ts|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|pickup_date|pickup_month|pickup_hour| trip_duration_min|PU_location_id|PU_borough|             PU_zone|DO_location_id|DO_borough|             DO_zone|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+---

In [60]:
# Group By -> SHUFFLE (COLUMNA)
# Ajustar particiones para reducir el overhead cuando tengo datasets pequenos o medianos < 100GB
# cache() para el reuso del mismo DF muchas veces

In [62]:
df_gold_viajes_por_zone_y_mes = (
    df_silver_enriched
    .groupby('pickup_month', 'PU_zone')
    .agg(
        F.count('*').alias('num_viajes'),
        F.avg('trip_distance').alias('avg_trip_distance')
    )
    .orderBy(F.desc('num_viajes'))
)

In [63]:
df_gold_viajes_por_zone_y_mes.show()

+------------+--------------------+----------+------------------+
|pickup_month|             PU_zone|num_viajes| avg_trip_distance|
+------------+--------------------+----------+------------------+
|     2024-01|      Midtown Center|    141829|2.5703397048558516|
|     2024-01|Upper East Side S...|    141317|1.7018326174487126|
|     2024-01|         JFK Airport|    141182|15.638710104687577|
|     2024-01|Upper East Side N...|    135407|1.8519551426440328|
|     2024-01|        Midtown East|    105503|2.2351247831815164|
|     2024-01|Times Sq/Theatre ...|    104683|2.9243368073135088|
|     2024-01|Penn Station/Madi...|    103296| 2.278204577137541|
|     2024-01| Lincoln Square East|    103038|2.0935962460451423|
|     2024-01|   LaGuardia Airport|     88579|  9.61596292574992|
|     2024-01|Upper West Side S...|     87720| 2.262874145006827|
|     2024-01|       Midtown North|     84665|  2.51198405480424|
|     2024-01|         Murray Hill|     82161| 2.198156789717759|
|     2024

In [66]:
# Ventas totales y tips por zone y mes

gold_revenue_by_zone_mes = (
    df_silver_enriched
    .groupby('pickup_month', 'PU_zone')
    .agg(
        F.count('*').alias('num_trips'),
        F.sum('total_amount').alias('total_revenue'),
        F.sum('fare_amount').alias('total_fare'),
        F.sum('tip_amount').alias('total_tip')
    )
    .withColumn('tip_pct', F.col('total_tip') /  F.when(F.col('total_fare') == 0, None).otherwise(F.col('total_fare')))
    .orderBy('pickup_month', F.desc('total_revenue'))
)

In [67]:
gold_revenue_by_zone_mes.show()

+------------+--------------------+---------+--------------------+------------------+------------------+-------------------+
|pickup_month|             PU_zone|num_trips|       total_revenue|        total_fare|         total_tip|            tip_pct|
+------------+--------------------+---------+--------------------+------------------+------------------+-------------------+
|     2002-12|         Murray Hill|        1|                10.5|               6.5|               0.0|                0.0|
|     2009-01|   LaGuardia Airport|        1|               68.29|              50.6|               0.0|                0.0|
|     2009-01|Upper East Side S...|        1|                50.0|              45.0|               0.0|                0.0|
|     2009-01|            Kips Bay|        1|                 9.4|               4.4|               0.0|                0.0|
|     2023-12|Sutton Place/Turt...|        1|               45.72|              33.1|              7.62|0.23021148036253775|


In [68]:
gold_revenue_by_zone_mes.explain(True)

== Parsed Logical Plan ==
'Sort ['pickup_month ASC NULLS FIRST, 'total_revenue DESC NULLS LAST], true
+- Project [pickup_month#397, PU_zone#660, num_trips#2578L, total_revenue#2580, total_fare#2582, total_tip#2584, (total_tip#2584 / CASE WHEN (total_fare#2582 = cast(0 as double)) THEN cast(null as double) ELSE total_fare#2582 END) AS tip_pct#2591]
   +- Aggregate [pickup_month#397, PU_zone#660], [pickup_month#397, PU_zone#660, count(1) AS num_trips#2578L, sum(total_amount#156) AS total_revenue#2580, sum(fare_amount#150) AS total_fare#2582, sum(tip_amount#153) AS total_tip#2584]
      +- Join LeftOuter, (DOLocationID#148 = DO_location_id#664)
         :- Join LeftOuter, (PULocationID#147 = PU_location_id#658)
         :  :- Filter (trip_duration_min#442 >= cast(0 as double))
         :  :  +- Filter (total_amount#156 >= cast(0 as double))
         :  :     +- Filter (trip_distance#144 >= cast(0 as double))
         :  :        +- Project [VendorID#140, pickup_ts#336, dropoff_ts#356, pas

In [70]:
gold_revenue_by_zone_mes.write.mode('overwrite').partitionBy('pickup_month').parquet('data/gold_trips/months')